# DeconvnetPreprocessor Test

- Creation : *04/04/2024*
- Mise à jour : *16/12/2024*

Chargement d'un modèle préentrainé (AlexNet, VGG16, ...) et visualisation des activations intermédiaires (dont les cartes de caractéristiques).

## Modules

In [1]:
import os
import json
import math
from functools import partial
from typing import Any, Tuple, List

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.utils import make_grid
import torchvision.transforms as T
import torchvision.transforms.functional as F
from torchvision.io import read_image

from torchinfo import summary
#plt.rcParams["savefig.bbox"] = 'tight'

## Device

In [2]:
################@
def get_best_device():
    if torch.cuda.is_available():
        # move models to GPU
        device = torch.device("cuda")
        print('CUDA GPU available for training. Models moved to CUDA GPU')
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print('MPS GPU available for training. Models moved to MPS GPU')
    else:
        device = torch.device("cpu")
        print('Training on CPU')
    return device
device = get_best_device()

MPS GPU available for training. Models moved to MPS GPU


## Deconvnet

### `get_receptive_field()`, `get_receptive_field_conv2d()`, `get_receptive_field_pool2d()`

In [ ]:
def get_receptive_field(pos, k, s, p):
    _to_tuple = lambda v: (v, v)
    if type(k) == int:
        k = _to_tuple(k)
    if type(s) == int:
        s = _to_tuple(s)
    if type(p) == int:
        p = _to_tuple(p)
    column = -p[1] + pos[1] * s[1]
    row = -p[0] + pos[0] * s[0]
    return (row, column), (row+k[1], column+k[0])

def get_receptive_field_conv2d(pos, k, s, p):
    return get_receptive_field(pos, k, s, p)

def get_receptive_field_pool2d(pos, k, s, p): 
    return get_receptive_field(pos, k, s, p)

### Classe `DeconvnetProcessor`

In [ ]:
class DeconvnetProcessor(CNNFeaturesHandler):
    def __init__(self, cnn_model_features: list[nn.Module]|nn.Sequential):
        """ New instance initialization """
        super().__init__(cnn_model_features)
        self.reset()


    def update_model(self, new_status=True) -> None:
        """
            Set MaxPooling layer's return_indices setting to True in order to get indices necessary for deconv backward
        """
        for layer in self.model_features:
            if isinstance(layer, nn.MaxPool2d):
                layer.return_indices=new_status


    def reset(self) -> None:
        self.maxpool_indices = []
        self.forward_outputs_ = None
        self.strongest_activation_ = None # will be (float, [int, int])


    def get_strongest_activation_feature_map(self, feature_map: torch.Tensor) -> torch.Tensor:
        """ 
        """
        o = torch.zeros_like(feature_map)
        idx_max = torch.argmax(feature_map).item()
        row_max = idx_max // feature_map.size(0)
        column_max = idx_max % feature_map.size(1)
        max = feature_map.max()
        self.strongest_activation_ = (max, (row_max, column_max))
        o[row_max, column_max] = self.strongest_activation_[0]
        return o
        

    def get_feature_maps_for_backward(self, feature_maps: torch.tensor, kernel_idx: int=0) -> torch.Tensor:
        """ Returns the zeroed output tensor but the channel_idx'th feature map """
        t = torch.zeros_like(feature_maps)
        strongest_activation_feature_map = self.get_strongest_activation_feature_map(feature_maps[0, kernel_idx])
        t[0, kernel_idx] = strongest_activation_feature_map
        return t


    def forward_keeping_indices(self, x: torch.Tensor, to_layer_idx: int=-1, verbose=False) -> torch.Tensor:
        """ 
        - Argument(s) :
            x : torch.Tensor
                size = (batch, channels, H, W)
            to_layer_idx : int
        """
        to_layer_idx = self.get_normalized_idx(to_layer_idx)
        self.assert_correct_layer_idx(to_layer_idx)

        self.update_model(new_status=True) # Must be done before .val()
        self.model_features.eval()
        with torch.no_grad():
            self.maxpool_indices.clear()
            for idx, layer in enumerate(self.model_features):
                print(idx, layer)

                if isinstance(layer, nn.MaxPool2d):
                    x, indices = layer(x)
                    self.maxpool_indices.append(indices)
                else:
                    x = layer(x)
                # stop at the to_layer_idx'th layer
                if idx == to_layer_idx:
                    break;
        self.model_features.train()
        self.update_model(new_status=False)
        return x


    def backward(self, y: torch.Tensor, from_layer_idx: int=-1, flip_kernel=False, verbose=False):
        """
        Because the deconvnet process is a symetric of the feature part of the CNN model
        """
        from_layer_idx = self.get_normalized_idx(from_layer_idx)
        self.assert_correct_layer_idx(from_layer_idx)

        idx_maxpool_indices = -1
        for i, layer in enumerate(reversed(self.model_features), start=1):
            idx = len(self.model_features) - i

            # only backward from the from_layer_idx'th layer
            if from_layer_idx < idx:
                continue

            if verbose:
                print(f"idx :{idx}", f"y : {y.size()}", layer)

            if isinstance(layer, nn.MaxPool2d):
                indices = self.maxpool_indices[idx_maxpool_indices]
                y = nn.functional.max_unpool2d(
                    y,
                    indices=indices,
                    kernel_size=layer.kernel_size,
                    stride=layer.stride,
                    padding=layer.padding
                )
                idx_maxpool_indices -= 1
            
            elif isinstance(layer, nn.ReLU):
                y = nn.functional.relu(y)
            
            elif isinstance(layer, nn.Conv2d):
                weights = self.model_features[idx].weight
                # No need to transpose weights kernel tensor
                #weights = torch.transpose(weights, 0, 1)
                if flip_kernel:
                    torch.flip(weights, [2, 3])

                if verbose:
                    print("weights: ", weights.size())

                y = nn.functional.conv_transpose2d(
                    y,
                    weight=weights,
                    #bias=layer.bias,
                    stride=layer.stride,
                    padding=layer.padding,
                    output_padding=1 if layer.stride[0] > 1 else 0, # Because stride > 1
                    dilation=layer.dilation
                )
        
        return y


    def forward_backward(self, 
                         x: torch.Tensor, 
                         to_layer_idx: int=-1, 
                         kernel_idx: int=0, 
                         flip_kernel=False, 
                         reset_forward=True,
                         verbose=False) -> torch.Tensor:
        """ """
        if reset_forward or self.forward_outputs_ == None:
            self.reset()
            self.forward_outputs_ = self.forward_keeping_indices(x, to_layer_idx, verbose=verbose)

        y = self.get_feature_maps_for_backward(self.forward_outputs_, kernel_idx)
        y = self.backward(y, to_layer_idx, flip_kernel=flip_kernel, verbose=verbose)
        
        return y
    

    def get_receptive_field_of_activation(self, idx_layer: int, activation_pos: Tuple[int, int]):
        """ """
        idx_layer = self.get_normalized_idx(idx_layer)
        self.assert_correct_layer_idx(idx_layer)
        
        tlc, brc = activation_pos, activation_pos
        for module in reversed(self.model_features[:idx_layer+1]):
            #print(module)
            if isinstance(module, nn.Conv2d):
                tlc, _ = get_receptive_field_conv2d(tlc, module.kernel_size, module.stride, module.padding)
                _, brc = get_receptive_field_conv2d(brc, module.kernel_size, module.stride, module.padding)
            elif isinstance(module, nn.MaxPool2d):
                tlc, _ = get_receptive_field_pool2d(tlc, module.kernel_size, module.stride, module.padding)
                _, brc = get_receptive_field_pool2d(brc, module.kernel_size, module.stride, module.padding)
            #print(tlc, brc)
        return (tlc, brc)

#### Tests de `DeconvnetProcessor`

##### `.__init__()`

In [ ]:
model = models.alexnet(weights='IMAGENET1K_V1')
model_features = model.features
deconv = DeconvnetProcessor(model_features)
deconv.show_feature_list()

In [ ]:
# !! Warning !! maxpooling return_indices setting must be False
input_batch_size = torch.Size([1, 3, 224, 224])
s = summary(deconv.model_features, input_size=input_batch_size)
s

In [ ]:
s.summary_list[1].output_size

In [ ]:
deconv.summary()

##### `.get_strongest_activation_feature_map()`

In [ ]:
to_layer_idx = 5
feature_maps, _ = features_handler.get_feature_layer_output(input_batch, to_layer_idx)
feature_maps = feature_maps.squeeze()
kernel_idx = feature_maps.size(0) // 2
feature_map = feature_maps[kernel_idx]
print(kernel_idx, feature_map.size())
strongest_activation_feature_map = deconv.get_strongest_activation_feature_map(feature_map)
print(strongest_activation_feature_map.max(), feature_map.max())
assert strongest_activation_feature_map.max() == feature_map.max(), "Issue with get_strongest_activation_feature_map"
assert deconv.strongest_activation_[0] == feature_map.max(), "Issue with get_strongest_activation_feature_map and strongest_activation_"

##### `.get_feature_maps_for_backward()`

In [ ]:
t = torch.rand(1, 3, 5, 5)
t = torch.nn.functional.normalize(t)
for i in range(3):
    t[0, i, i, i] = i + 2

t_t = torch.zeros_like(t)
t_t[0, 0, 0, 0] = 2
fmfb = deconv.get_feature_maps_for_backward(t, 0)
assert torch.equal(fmfb.type(torch.float16), t_t.type(torch.float16)), "Issue with get_strongest_activation_feature_map"

t_t = torch.zeros_like(t)
t_t[0, 1, 1, 1] = 3
assert torch.equal(deconv.get_feature_maps_for_backward(t, 1), t_t), "Issue with get_strongest_activation_feature_map"

t_t = torch.zeros_like(t)
t_t[0, 2, 2, 2] = 4
assert torch.equal(deconv.get_feature_maps_for_backward(t, 2), t_t), "Issue with get_strongest_activation_feature_map"

##### `.update_model()`

In [ ]:
input = torch.randn(20, 16, 50, 32)

deconv.update_model(new_status=True)
maxpool_layer = deconv.get_feature_layer(2)
#print(maxpool_layer)
output = maxpool_layer(input)
assert len(output)==2, "Issue with update_model (True)"

deconv.update_model(new_status=False)
maxpool_layer = deconv.get_feature_layer(2)
output = maxpool_layer(input)
#print(output)
assert isinstance(output, torch.Tensor), "Issue with update_model (False)"

##### `.forward_keeping_indices()`

In [ ]:
to_layer_idx = 6
forward_output = deconv.forward_keeping_indices(input_batch, to_layer_idx=to_layer_idx)
assert forward_output.size() == torch.Size([1, 384, 13, 13]), "Issue with forward_keeping_indices"

In [ ]:
to_layer_idx = -1
forward_output = deconv.forward_keeping_indices(input_batch, to_layer_idx=to_layer_idx)
assert forward_output.size() == torch.Size([1, 256, 6, 6]), "Issue with forward_keeping_indices"

##### `.backward()`

In [ ]:
to_layer_idx = 2
forward_output = deconv.forward_keeping_indices(input_batch, to_layer_idx=to_layer_idx)
print(forward_output.size())
assert forward_output.size() == torch.Size([1, 64, 27, 27]), "Issue with forward_keeping_indices"

In [ ]:
channel_idx = forward_output.size(1) // 2
print(channel_idx)
input_backward = deconv.get_feature_maps_for_backward(forward_output, channel_idx)
print(input_backward.size())
assert input_backward.size() == torch.Size([1, 64, 27, 27]), "Issue with forward_keeping_indices"
assert input_backward[0, 0, :, :].sum().item() == 0, "Issue with forward_keeping_indices"
assert input_backward[0, -1, :, :].sum().item() == 0, "Issue with forward_keeping_indices"

In [ ]:
from_layer_idx = to_layer_idx
backward_output = deconv.backward(input_backward, from_layer_idx)
backward_output.size()

##### `.forward_backward()`

In [ ]:
to_layer_idx = 3 # After the first Conv2D-ReLU-MaxPool block
number_channels = deconv.get_feature_layer_summary(to_layer_idx)[1]
output_backward = deconv.forward_backward(input_batch, to_layer_idx, channel_idx)
print(output_backward.size(1))
assert output_backward.size(1) == 3, "Issue with forward_backward"

##### `.get_receptive_field_of_activation()`

In [ ]:
deconv.get_receptive_field_of_activation(idx_layer=3, activation_pos=(1, 1))

#### `shift_to_RGB_range()`

In [ ]:
def shift_to_RGB_range(t: np.array, range256: bool=False, min: int|float=None, max: int|float=None) ->  torch.Tensor:
    """ 
    Args:
        t : torch.Tensor
            (n, c, H, W) tensor of images
        range256 : bool
            range [0, 255] if True, range [0, 1.0] else
    """
    shift_min = 0
    shift_max = 256 if range256 else 1.
    t_min = t.min() if min == None else min
    t_max = t.max() if max == None else max

    shifted = (shift_max - shift_min) * (t - t_min) / (t_max - t_min) + shift_min
    shifted = shifted.astype(np.uint8) if range256 else shifted.astype(np.float32)

    return shifted

#### `tensor3d_to_image()`

In [ ]:
def tensor3d_to_image(t: torch.Tensor) -> np.array:
    npimg = t.detach().numpy()
    npimg = np.transpose(npimg, (1, 2, 0))
    #npimg = ((npimg * imagenet_std) + imagenet_mean)

    return npimg

#### `show_tensor3d_as_image()`

In [ ]:
def show_tensor3d_as_image(t: torch.Tensor, axis: bool=False, title: str="", shift=True):
    npimg = tensor3d_to_image(t)
    if shift:
        npimg = shift_to_RGB_range(npimg)

    if not axis:
        plt.axis("off")
    if title:
        plt.title(title)
    plt.imshow(npimg, interpolation="nearest");
    plt.show()

In [ ]:
t = backward_output.squeeze()
print(t.min().item(), t.max().item())

In [ ]:
show_tensor3d_as_image(t.squeeze())

In [ ]:
show_tensor3d_as_image(input_batch.squeeze())

## Noyaux de la première couche

In [ ]:
deconv.get_conv_layer(1)

In [ ]:
deconv.get_conv_def(1)

In [ ]:
deconv.get_feature_layer_summary(deconv.get_conv_index(1))

In [ ]:
weights = deconv.get_conv_layer(1).weight
weights.size()

In [ ]:
weights[0]

In [ ]:
weights = weights.detach().numpy()
w = weights[0]
print(w.min(), w.max(), w.mean(), w.std())

In [ ]:
weights_r = shift_to_RGB_range(weights)
weights_r

In [ ]:
weights_r.size()

### `display_grid_3c()`

In [ ]:
def display_grid_3c(
        pictures: np.array,
        images_per_row: int=16,
        title: str="",
        cmap: str="viridis",
        figsize: Tuple[int,int]=None
        ) -> None:
    """ 
    - Argument(s)
        images_per_row : int
            Nombre d'images par ligne
    """
    
    ##DEBUG
    #print(pictures.size())

    # Nombre de canaux de la carte d'activation
    # And taille spatiale d'une carte présumée carrée
    n, size, _, _ = pictures.shape
    # Nombre de lignes nécessaires
    n_cols = math.ceil(n / images_per_row)

    # Taille de la grille avec séparation de 1 pixel en H/Y et W/X
    margin_H, margin_W = 1, 1
    
    ##DEBUG
    #print(n_features, size, n_cols)
    #print(((size + margin_W) * n_cols - margin_W,
    #                        images_per_row * (size + margin_H) - margin_H))
    
    display_grid = np.zeros(((size + margin_W) * n_cols - margin_W,
                            images_per_row * (size + margin_H) - margin_H,
                            3))
    # Pour chaque colonne
    for col in range(n_cols):
        # Pour chaque ligne
        for row in range(images_per_row):
            # Index du canal
            channel_index = col * images_per_row + row
            if channel_index >= len(pictures):
                break
            # Matrice de pixels de ce canal
            channel_image = pictures[channel_index, :, :, :]
            # Application de l'image dans la grille
            display_grid[
                col * (size + 1): (col + 1) * size + col,
                row * (size + 1) : (row + 1) * size + row, :] = channel_image
        
    # Mise à l'échelle pour l'affichage
    if not figsize:
        scale = 1. / size
        figsize = (scale * display_grid.shape[1], scale * display_grid.shape[0])
    plt.figure(figsize=figsize);
    
    # Affichage
    if title:
        plt.title(title)
    plt.grid(False)
    plt.axis("off")
    plt.imshow(display_grid, aspect="auto", cmap=cmap)

In [ ]:
display_grid_3c(
    torch.permute(weights_r.detach(), (0, 2, 3, 1)).numpy(),
    images_per_row=10,
    title="Poids de noyaux de la première couche"
    )

In [ ]:
image_array = shift_to_RGB_range(weights_r[])

R = image_array[:,:,0]
G = image_array[:,:,1]
B = image_array[:,:,2]

bin_edges = np.linspace(0, 255, 256)
hist_R, _ = np.histogram(R, bins=bin_edges, density=True)
hist_G, _ = np.histogram(G, bins=bin_edges, density=True)
hist_B, _ = np.histogram(B, bins=bin_edges, density=True)

# Créer une figure pour les courbes
plt.figure(figsize=(10, 6))

# Afficher les courbes
plt.step(bin_edges[:-1], hist_R, where='mid', color='red', label='Rouge')
plt.step(bin_edges[:-1], hist_G, where='mid', color='green', label='Vert')
plt.step(bin_edges[:-1], hist_B, where='mid', color='blue', label='Bleu')

# Ajouter des légendes et des titres
plt.title('Distribution des canaux RGB')
plt.xlabel('Intensité de couleur')
plt.ylabel('Densité')
plt.legend()

# Afficher le graphique
plt.show()

## Visualisation des caractérisques de la première couche

In [ ]:
model = models.alexnet(weights='IMAGENET1K_V1')
model_features = model.features
deconv = DeconvnetProcessor(model_features)
deconv.summary(verbose=False)

In [ ]:
input_batch_size[2]

In [ ]:
idx_conv = deconv.get_conv_index(2)
to_layer_idx = idx_conv + 2
number_kernels = deconv.get_feature_layer_summary(to_layer_idx)[1]
outputs = []
print(f"to_layer_idx: {to_layer_idx} , number_kernels: {number_kernels}")

number_kernels = 1
for kernel_idx in range(number_kernels):
    output_backward = deconv.forward_backward(input_batch, to_layer_idx, kernel_idx, reset_forward=False, flip_kernel=False)
    output_backward_size = output_backward.size()
    #show_tensor3d_as_image(output_backward.squeeze())
    array = torch.clamp(output_backward.detach().squeeze(), min=0).numpy()
    print(f"[{kernel_idx}]", array.shape)
    print("\t", deconv.strongest_activation_)
    
    # Crop to get image patch corresponding of receptive field of max activation
    tlc, brc = deconv.get_receptive_field_of_activation(to_layer_idx, activation_pos=deconv.strongest_activation_[1])
    tlc = list(tlc)
    brc = list(brc)
    # The field must be in picture space
    if tlc[0] < 0:
        brc[0] -= tlc[0]
        tlc[0] = 0
    if tlc[1] < 0:
        brc[1] -= tlc[1]
        tlc[1] = 0
    if brc[0] >= output_backward_size[2]:
        tlc[0] -= brc[0] - (output_backward_size[2] - 1)
        brc[0] = output_backward_size[2] - 1
    if brc[1] >= output_backward_size[3]:
        tlc[1] -= brc[1] - (output_backward_size[3] - 1)
        brc[1] = output_backward_size[3] - 1

    print("\t", kernel_idx, array.min(), array.max(), array.mean(), tlc, brc, np.array(brc) - np.array(tlc))

    picture = shift_to_RGB_range(output_backward.detach()).squeeze()
    #picture = torch.clamp(output_backward.detach(), min=0).squeeze()
    #picture = shift_to_RGB_range(torch.clamp(output_backward.detach(), min=0).squeeze())

    picture = torch.permute(picture, (1, 2, 0)).numpy()
    print("\tshifted picture.shape", picture.shape, type(picture))

    cropped_picture = picture[tlc[0]:brc[0]+1, tlc[1]:brc[1]+1, :]
    #cropped_picture = picture
    print("\tcropped_picture.shape :", cropped_picture.shape)
    outputs.append(cropped_picture)

outputs = np.array(outputs)
print(outputs.shape)
display_grid_3c(outputs, images_per_row=10, title=f"Caractéristiques après la couche {to_layer_idx + 1} (base 1)")

In [ ]:
plt.imshow(picture)
plt.axis("on")
plt.show()

In [ ]:
deconv.forward_outputs_[0, 0].max()

In [ ]:
p = deconv.forward_outputs_[0, 0].detach().numpy()
print(p.shape)
plt.imshow(p)
plt.axis("on")
plt.colorbar()
plt.show()

In [ ]:
deconv.get_receptive_field_of_activation(idx_layer=2, activation_pos=(16, 16))

In [ ]:
to_layer_idx